# Notebook 02 — Exposure Extraction and Linkage
**GeoAI for Climate, Environment and Health**

---

> **Where you are in the pipeline**
>
> ```
> 📋 Health records → [🛰 Satellite + 🔗 Link] → ⏱ Timing → 🎯 Results & GeoAI
> ```
>
> You have birth records with geographic locations — DHS cluster centroids
> scattered across Zimbabwe. Before any modelling is possible you need to
> **attach environmental values to those locations**. This notebook covers
> both steps that achieve that:
>
> 1. **Extraction** — pulling satellite and reanalysis measurements for each
>    cluster location using Google Earth Engine.
> 2. **Linkage** — spatially joining the extracted climate data to individual
>    birth records so every row in the health dataset carries the environment
>    its mother was exposed to.
>
> By the end of this notebook you will understand where the numbers in the
> pre-built climate dataset come from and how they are connected to real births.

---

In [ ]:
%%html
<div id="prereq-gate-html" style="font-family:system-ui,sans-serif;margin:0 0 8px"></div>
<script>
(function(){
  var key="nb01_passed",label="Notebook 01: Spatial Health Data",fn="01_spatial_health_data.ipynb";
  var g=document.getElementById("prereq-gate-html");
  if(!g)return;
  if(!localStorage.getItem(key)){
    g.innerHTML='<div style="background:#f8d7da;border:2px solid #dc3545;border-left:8px solid #dc3545;border-radius:8px;padding:18px 22px;color:#721c24">'
      +'<div style="font-size:20px;font-weight:800;margin-bottom:8px">⛔ STOP — Prerequisites Not Met</div>'
      +'<p style="margin:0 0 10px;font-size:15px">You have not yet passed the quiz for <b>'+label+'</b>. Running this notebook now means you will miss the foundations it builds on.</p>'
      +'<p style="margin:0;font-size:14px"><b>What to do:</b> Open <code>'+fn+'</code>, work through it, pass the quiz, and run the <b>Save Progress</b> cell at the bottom. Then return here.</p>'
      +'</div>';
  }else{
    g.innerHTML='<div style="background:#d1e7dd;border-left:5px solid #198754;border-radius:6px;padding:10px 16px;color:#0a3622"><b>✓ '+label+' completed</b> — welcome to the next module.</div>';
  }
})();
</script>

In [ ]:
import pathlib as _pl
# ── Prerequisite check — run before anything else ─────────────────────────
_badge = _pl.Path.home() / '.nb01_complete'
if not _badge.exists():
    print("⛔  STOP — You must complete Notebook 01: Spatial Health Data before this notebook.")
    print()
    print("   Steps:")
    print("   1. Open 01_spatial_health_data.ipynb")
    print("   2. Work through the notebook and pass the quiz")
    print("   3. Run the 'Save Progress' cell at the bottom")
    print("   4. Return here and re-run this cell")
    raise SystemExit("Prerequisites not met.")
print("✅  Prerequisite check passed — welcome to this module.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy import stats
import matplotlib.ticker as mticker
import requests
import pathlib, sys

def _find_data():
    here = pathlib.Path().resolve()
    for p in (here, here.parent, here.parent.parent):
        if (p / 'data').is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p / 'data'
    raise FileNotFoundError('Cannot find data/ directory')

DATA_DIR = _find_data()
print(f'Data directory: {DATA_DIR}')

def mc_question(question, options, correct, explanation):
    try:
        import ipywidgets as w
        from IPython.display import display, HTML
        header = w.HTML(f'<div style="font-size:15px;font-weight:600;margin-bottom:10px">🧠 {question}</div>')
        radio = w.RadioButtons(options=[f"{k})  {v}" for k, v in options.items()],
                               layout=w.Layout(width="100%"),
                               style={"description_width": "0px"})
        feedback = w.Output()
        def _check(change):
            if change["name"] != "value": return
            letter = change["new"][0]
            with feedback:
                feedback.clear_output(wait=True)
                if letter == correct:
                    display(HTML(f'<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #28a745">✅ <b>Correct!</b>&nbsp; {explanation}</div>'))
                else:
                    display(HTML(f'<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b>&nbsp; The answer is <b>{correct}</b>. {explanation}</div>'))
        radio.observe(_check, names="value")
        display(w.VBox([header, radio, feedback],
                       layout=w.Layout(border="1px solid #dee2e6", padding="16px 20px",
                                       border_radius="8px", margin="16px 0")))
    except ImportError:
        print(f"Q: {question}")
        for k, v in options.items():
            print(f"  {k}) {v}{' ← CORRECT' if k == correct else ''}")
        print(f"Explanation: {explanation}")

---
## Section 1 — Why Satellite Data? What We Extract and Why

Health surveys like DHS collect where births happen but not what the
environment looked like. Satellites fill that gap: they measure the land
surface continuously, globally, and for free — making them the backbone
of environmental epidemiology in low-resource settings.

| Variable | Product | Resolution | What it measures |
|---|---|---|---|
| `lst_*` | MODIS MOD11A2 | 1 km / 8-day | Daytime land surface temperature |
| `rain_*` | CHIRPS v2 | ~5 km / monthly | Precipitation |
| `ndvi_*` | MODIS MOD13A3 | 500 m / monthly | Vegetation greenness (NDVI) |
| `heat_*` | ERA5-Land | ~9 km / hourly | 2 m air temperature + humidity |

**Key design consideration — DHS displacement.** DHS displaces cluster
coordinates by up to 5 km (urban) or 10 km (rural) to protect anonymity.
A 500 m MODIS pixel may not overlap the true cluster location, but a 25 km
ERA5 grid cell almost certainly still contains it. Coarser resolution products
are therefore *more robust* to this displacement — a counterintuitive but
important point for study design.

**All exposures are area-level.** They capture ambient environment at the
cluster centroid, not individual personal exposure. The ecological fallacy
applies and within-cluster heterogeneity is invisible to the model.

---
## Section 2 — Earth Engine Extraction: One Live Example

**How the pre-built dataset was created.** The conceptual workflow below
shows exactly how the extraction was run for all 399 clusters and 60 months.
You do not need a GEE account — the key ideas are:

1. Upload cluster coordinates as a `FeatureCollection`.
2. Load the satellite image collection and filter to your date range.
3. For each image, sample the value at each cluster centroid
   (`reduceRegions`).
4. Export to CSV on Google Drive.

```python
import ee
ee.Initialize()

# 1. Cluster coordinates uploaded as an EE asset
clusters_fc = ee.FeatureCollection('users/your_username/zw_dhs_clusters')

# 2. MODIS LST — scale from raw digital numbers → °C
modis_lst = (
    ee.ImageCollection('MODIS/061/MOD11A2')
    .filterDate('2011-01-01', '2015-12-31')
    .select('LST_Day_1km')
    .map(lambda img: img.multiply(0.02).subtract(273.15))  # K → °C
)

# 3. Extract value at each centroid for every image in the collection
def sample_image(img):
    return img.reduceRegions(
        collection=clusters_fc,
        reducer=ee.Reducer.mean(),
        scale=1000
    ).map(lambda f: f.set('date', img.date().format('YYYY-MM')))

sampled = modis_lst.map(sample_image).flatten()

# 4. Export
ee.batch.Export.table.toDrive(
    collection=sampled,
    description='zw_modis_lst_2011_2015',
    fileFormat='CSV'
).start()
```

The `zw_climate_monthly.csv` we use in this workshop is the result of this
workflow run for LST (MODIS), rainfall (CHIRPS), NDVI (MODIS), and heat
index (ERA5) across all 399 clusters for 60 months.

In [ ]:
EE_API = 'http://localhost:5000/api/workshop/ee_query'

def ee_query(collection, band, lat, lon, date,
             scale=5000, value_scale=1.0, date_window_days=0):
    """Query EE via the server proxy — no authentication needed."""
    r = requests.post(EE_API, json={
        'collection': collection, 'band': band,
        'lat': lat, 'lon': lon, 'date': date,
        'scale': scale, 'value_scale': value_scale,
        'date_window_days': date_window_days,
    }, timeout=30)
    result = r.json()
    if 'error' in result:
        raise RuntimeError(result['error'])
    return result['value']

# Quick connection test
test = ee_query('UCSB-CHG/CHIRPS/DAILY', 'precipitation',
                lat=-17.303925, lon=31.888711, date='2022-12-15')
print(f'Earth Engine proxy ready ✓   test value: {test} mm')

### Live Query — Monthly NDVI for a DHS Cluster

NDVI (Normalised Difference Vegetation Index) ranges from −1 to +1;
dense green vegetation is ~0.5–0.8 and bare soil is ~0.1–0.2.
Zimbabwe has a strong wet/dry seasonal cycle, so NDVI is an informative
proxy for the agricultural and nutritional environment during pregnancy.

Below we query MODIS MOD13A1 (16-day composite, 500 m) for a single cluster
across all 12 months of 2013 — exactly what the pipeline does for all 399
clusters across 60 months.

In [ ]:
# Cluster 1: Mashonaland East, rural (-17.30°S, 31.89°E)
lat, lon = -17.303925, 31.888711

months = pd.date_range('2013-01-01', '2013-12-01', freq='MS')
ndvi_vals = []
for m in months:
    v = ee_query('MODIS/061/MOD13A1', 'NDVI',
                 lat=lat, lon=lon,
                 date=m.strftime('%Y-%m-%d'),
                 scale=500, value_scale=1e-4, date_window_days=32)
    ndvi_vals.append({'month': m.strftime('%b'), 'ndvi': v or 0})

df_ndvi = pd.DataFrame(ndvi_vals)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(df_ndvi['month'], df_ndvi['ndvi'], color='seagreen', alpha=0.8)
ax.axhline(0.3, color='orange', linestyle='--', label='Sparse vegetation threshold')
ax.set_ylabel('NDVI')
ax.set_title('Cluster 1 — MODIS NDVI monthly, 2013 (Mashonaland East, rural)')
ax.legend()
plt.tight_layout(); plt.show()

print("Wet season (Nov–Mar): high NDVI — crops growing, grazing land green.")
print("Dry season (May–Sep): NDVI drops sharply — vegetation senescence and bare soil.")

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">&#129504; The climate dataset provides monthly values at 399 DHS cluster locations. Why is climate data attached at the cluster level rather than to individual household locations?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q1" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> Earth Engine cannot process more than 399 location points at once</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q1" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> DHS household coordinates are intentionally jittered (displaced) for privacy — attaching climate at finer resolution than the location data would be false precision</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q1" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Monthly climate averages do not vary within a cluster area</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q1" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> Household-level extraction requires additional survey consent forms</span></label></div>
  <div id="nb02q1-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="DHS jitters cluster coordinates by up to 5 km (rural) to protect respondent identity. The cluster is the finest meaningful spatial unit — extracting climate at the cluster level matches the actual precision of the location data.",ID="nb02q1";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Section 3 — Load the Pre-extracted Climate Data

Running GEE live for all 399 clusters × 60 months would take several minutes
and requires a GEE account. We pre-ran the extraction and stored the results
in `zw_climate_monthly.csv`. The cell below loads all tables needed for the
spatial join that follows.

In [ ]:
# Load all data tables needed for exposure linkage
climate    = pd.read_csv(DATA_DIR / 'zw_climate_monthly.csv')
clusters   = pd.read_csv(DATA_DIR / 'zw_clusters.csv')
births     = pd.read_csv(DATA_DIR / 'zw_births_raw.csv')
facilities = pd.read_csv(DATA_DIR / 'health_facilities.csv')
facilities = facilities.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

print(f"Climate records : {len(climate):,}  ({climate['cluster_id'].nunique()} clusters)")
print(f"Clusters        : {len(clusters):,}")
print(f"Births          : {len(births):,}")
print(f"Facilities      : {len(facilities):,}")
print()
print("Climate columns:", list(climate.columns))

In [ ]:
# Annual mean climate per cluster — one representative row per cluster
# for building the spatial index. Trimester-specific windows come later.
clim_annual = (
    climate
    .groupby('cluster_id')[['temp_c', 'rain_mm', 'ndvi', 'dust_aod550', 'pm25']]
    .mean()
    .reset_index()
    .rename(columns={'temp_c': 'mean_temp', 'rain_mm': 'mean_rain',
                      'ndvi': 'mean_ndvi', 'dust_aod550': 'mean_dust', 'pm25': 'mean_pm25'})
)
clim_annual = clim_annual.merge(
    clusters[['cluster_id', 'cluster_lat', 'cluster_lon', 'province', 'urban']],
    on='cluster_id'
)
print(f"Exposure grid: {len(clim_annual)} clusters")
print(clim_annual[['mean_temp', 'mean_rain', 'mean_ndvi', 'mean_pm25']].describe().round(2))

---
## Section 4 — Linking Climate Data to Birth Records

We now have two datasets:
- **Climate data** — monthly measurements at 399 DHS cluster locations across Zimbabwe
- **Birth records** — each birth carries a `cluster_id` that tells us where the birth occurred

**How the join works:**

Each birth is matched to the climate row with the same `cluster_id`. In practice this is a simple table merge — like a spreadsheet VLOOKUP. The only complexity is that the climate data is stored as a spatial index (a tree structure) so the computer can find the nearest cluster location quickly when `cluster_id` values differ slightly between tables.

The map below shows the 399 cluster locations coloured by mean annual temperature. Clusters in the Zambezi Valley and Limpopo lowlands are hottest; the Eastern Highlands and central plateau are cooler — already a hint that geography shapes pregnancy environments.


In [ ]:
# Build the spatial index on cluster coordinates
cluster_coords = clim_annual[['cluster_lat', 'cluster_lon']].values
tree = cKDTree(cluster_coords)
lat_arr = clim_annual['cluster_lat'].values
lon_arr = clim_annual['cluster_lon'].values

def haversine_approx_km(lat1, lon1, lat2, lon2):
    """Flat-earth approximation. Accurate to <0.5% for distances <200 km."""
    mid_lat = np.radians((np.asarray(lat1) + np.asarray(lat2)) / 2.0)
    dlat_km = (np.asarray(lat2) - np.asarray(lat1)) * 111.0
    dlon_km = (np.asarray(lon2) - np.asarray(lon1)) * 111.0 * np.cos(mid_lat)
    return np.sqrt(dlat_km**2 + dlon_km**2)

try:
    import contextily as ctx
    from pyproj import Transformer as _T
    _proj = _T.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
    cx, cy = _proj.transform(clim_annual['cluster_lon'].values, clim_annual['cluster_lat'].values)
    fx, fy = _proj.transform(facilities['longitude'].values, facilities['latitude'].values)
    clim_annual = clim_annual.copy()
    clim_annual['_mx'] = cx; clim_annual['_my'] = cy
    fig, ax = plt.subplots(figsize=(9, 8))
    sc = ax.scatter(clim_annual['_mx'], clim_annual['_my'],
                    c=clim_annual['mean_temp'], cmap='RdYlBu_r',
                    s=28, alpha=0.85, zorder=3,
                    label=f'DHS cluster (n={len(clim_annual)})')
    plt.colorbar(sc, ax=ax, label='Mean annual temperature (°C)', shrink=0.65)
    ax.scatter(fx, fy, marker='*', s=200, color='black', zorder=5,
               edgecolors='white', linewidths=0.5,
               label=f'Health facility (n={len(facilities)})')
    ax.set_title('Zimbabwe DHS 2015 — DHS Clusters and Health Facilities\n'
                 'Coloured by mean annual temperature (ERA5)', fontsize=12)
    ax.set_axis_off(); ax.legend(loc='lower right', fontsize=10)
    try:
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=7)
    except Exception:
        pass
    plt.tight_layout(); plt.show()
except ImportError:
    fig, ax = plt.subplots(figsize=(9, 8))
    sc = ax.scatter(clim_annual['cluster_lon'], clim_annual['cluster_lat'],
                    c=clim_annual['mean_temp'], cmap='RdYlBu_r', s=28, alpha=0.85,
                    label=f'DHS cluster (n={len(clim_annual)})')
    plt.colorbar(sc, ax=ax, label='Mean annual temperature (°C)', shrink=0.65)
    ax.scatter(facilities['longitude'], facilities['latitude'],
               marker='*', s=200, color='black', zorder=5, edgecolors='white',
               linewidths=0.5, label=f'Health facility (n={len(facilities)})')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_title('Zimbabwe DHS 2015 — Clusters and Health Facilities', fontsize=12)
    ax.legend(loc='lower right', fontsize=10)
    plt.tight_layout(); plt.show()

print(f"Grid: {len(clim_annual)} DHS clusters | Each cluster will supply climate values to nearby births.")

---
## Section 5 — Joining Births to Climate Data

The cell below merges birth records to the climate table using `cluster_id` as the key — one climate row per cluster.

> **Why annual means for this exercise?**
> Annual averages give us one clean number per cluster — easy to reason about and a good introduction to the linkage concept. You can see at a glance how temperature and rainfall distribute across births.
>
> **The limitation:** A birth in October (Zimbabwe's peak heat) and one in April (end of rainy season) both get the same *annual* temperature even though their pregnancies unfolded in completely different climates. Notebook 03 solves this by computing trimester-specific exposures — matching each pregnancy's actual calendar windows to the climate at conception, T1, T2, and T3.

Run the cell below to perform the join.


In [ ]:
births_env = births.merge(
    clim_annual[['cluster_id', 'mean_temp', 'mean_rain', 'mean_ndvi', 'mean_pm25', 'mean_dust']],
    on='cluster_id',
    how='left'
)

unmatched = births_env['mean_temp'].isnull().sum()
print(f"Unmatched births (no climate record): {unmatched}")
assert unmatched == 0, "All births must match a cluster — check cluster_id values."

frac_hot_clusters = round(float((births_env['mean_temp'] > 18).mean()), 3)

print(f"Births joined successfully: {len(births_env):,} rows")
print(f"Fraction of births in clusters with mean annual temp > 18 °C: {frac_hot_clusters:.3f}")

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">&#129504; When climate data is matched to each birth using the cluster location, all births in the same cluster get identical climate values. What is the key limitation of this approach?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q2" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> The join only works correctly for rural clusters</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q2" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Births within the same cluster may actually experience different microclimates — the cluster assignment ignores within-cluster environmental variation</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q2" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> The join over-estimates temperature because of the jitter displacement</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q2" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> There is no limitation — cluster-level matching is the gold standard</span></label></div>
  <div id="nb02q2-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="Assigning one climate value per cluster ignores within-cluster variation. A household on a valley floor and one on a hillside may be in the same DHS cluster but experience different temperatures. This is called spatial misclassification and it typically causes us to underestimate the true effect of environment on health.",ID="nb02q2";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">&#129504; A researcher uses the annual mean temperature linked here to test whether higher temperatures increase LBW risk, but finds no significant effect. What is the MOST likely explanation?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q3" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> The nearest-neighbour spatial join must have introduced errors</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q3" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Temperature genuinely has no relationship with birth outcomes in Zimbabwe</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q3" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Annual averages hide seasonal variation — a baby whose T3 fell in October (peak heat) and one whose T3 fell in April (cool, rainy) both get the same annual value, washing out the signal</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb02q3" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> The sample of 5,800 births is too small for any regression to find an effect</span></label></div>
  <div id="nb02q3-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="C",E="Annual means are too coarse — they smooth over the seasonal signal that biology responds to. The next notebook generates trimester-specific exposures matched to each pregnancy's actual calendar windows. This is exactly why the daily dataset was constructed.",ID="nb02q3";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Recap

| Concept | Key point |
|---|---|
| Satellite products | MODIS LST → heat; CHIRPS → rainfall; MODIS NDVI → vegetation; ERA5 → air temperature |
| GEE workflow | Upload coordinates → filter ImageCollection → `reduceRegions` → export CSV |
| Spatial join | `cKDTree` nearest-neighbour: O(log n) lookup; assigns cluster climate to every birth |
| DHS displacement | Up to 10 km jitter — coarser grids (ERA5 ~25 km) are more robust than fine products (MODIS 500 m) |
| Exposure misclassification | Area-level assignment is an acknowledged limitation; it attenuates true associations toward null |
| What is missing | Annual means hide *when* during pregnancy each climate value applies — the next notebook fixes this |

By the end of this step every birth record has environmental exposure values
attached — temperature, rainfall, vegetation, and dust levels drawn from
satellite imagery and reanalysis. This is what makes GeoAI for health
possible: a single merged table where geography, climate, and health outcomes
sit side by side, ready for the machine-learning pipeline in Notebook 04.

---

**Next:** Notebook 03 reconstructs trimester-specific exposure windows from
conception and birth dates, so the *timing* of environmental stress during
pregnancy is captured.

In [ ]:
%%html
<div id="nb02-scoreboard" style="font-family:system-ui,sans-serif;border:2px solid #adb5bd;border-radius:10px;padding:20px 24px;margin:24px 0;background:#f8f9fa">
  <div style="font-size:17px;font-weight:700;margin-bottom:12px;color:#1a1a2e">&#128202; Module 2 — Exposure Extraction and Linkage &#8212; Quiz Results</div>
  <div id="nb02-score-display"><p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Complete all reflection questions above to see your score.</p></div>
  <div id="nb02-next-action" style="margin-top:14px"></div>
</div>
<script>
(function(){
  var QUESTIONS=[{id:"nb02q1",correct:"B"},{id:"nb02q2",correct:"B"},{id:"nb02q3",correct:"C"}];
  var PASS=67;
  var NEXT="03_exposure_timeline_characterisation.ipynb";
  var NEXT_LABEL="Notebook 03: Exposure Timeline and Characterisation";
  var NBID="nb02";
  function update(){
    var correct=0,answered=0;
    QUESTIONS.forEach(function(q){
      var ans=localStorage.getItem("mcq_"+q.id);
      if(ans!==null){answered++;if(ans===q.correct)correct++;}
    });
    var total=QUESTIONS.length;
    var pct=total?Math.round(correct/total*100):0;
    var passed=pct>=PASS;
    var disp=document.getElementById(NBID+"-score-display");
    var act=document.getElementById(NBID+"-next-action");
    if(!disp)return;
    if(answered<total){
      disp.innerHTML='<p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Answer all '+total+' questions to see your score. ('+( total-answered)+' remaining)</p>';
      if(act)act.innerHTML="";
    }else if(passed){
      disp.innerHTML='<div style="background:#d4edda;color:#155724;padding:12px 16px;border-radius:8px"><b>&#10003; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; MODULE PASSED</b><br><small>Scroll down and run the <b>Save Progress</b> cell to unlock the next notebook.</small></div>';
      if(act){
        localStorage.setItem(NBID+"_passed","true");
        if(NEXT){act.innerHTML='<div style="margin-top:10px"><a href="'+NEXT+'" style="display:inline-block;background:#198754;color:#fff;padding:10px 22px;border-radius:6px;text-decoration:none;font-weight:600;font-size:14px">&#10145; Continue to '+NEXT_LABEL+'</a><div style="margin-top:8px;font-size:12px;color:#6c757d">&#128269; Also run the <b>Save Progress</b> cell below — it unlocks the prereq gate in the next notebook.</div></div>';}
      }
    }else{
      disp.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:12px 16px;border-radius:8px"><b>&#10007; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; Below pass mark ('+PASS+'%)</b><br><small>Go back to any question and change your answer.</small></div>';
      if(act)act.innerHTML="";
    }
  }
  document.addEventListener("mcq-update",update);
  if(document.readyState==="loading")document.addEventListener("DOMContentLoaded",update);
  else update();
  setInterval(update,3000);
})();
</script>

In [ ]:
import pathlib as _pl
# ── Save progress — run this after passing the quiz above ─────────────────
_badge = _pl.Path.home() / '.nb02_complete'
_badge.touch()
print(f"✅  Progress saved: {_badge}")
print("   You can now open Notebook 03: Exposure Timeline and Characterisation.")